# Clase 4 · Notebook 02
# Clasificador multiclase con softmax (Iris)

**Introducción al Deep Learning — Módulo 2, Unidad 2: Clasificación**

Ahora construiremos un clasificador **multiclase**: tres especies de flor del dataset **Iris**.
La diferencia clave con el caso binario está en la **salida** y la **pérdida**:

- Capa de salida con **n neuronas softmax** (una probabilidad por clase, suman 1).
- Pérdida **categorical crossentropy**.
- Etiquetas en formato **one-hot**.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Preparar los datos

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
tf.random.set_seed(42); np.random.seed(42)

iris = load_iris()
X = StandardScaler().fit_transform(iris.data)   # 4 atributos
y = iris.target                                 # 0, 1, 2 (tres especies)

# One-hot de la clase: 0 -> [1,0,0], 1 -> [0,1,0], 2 -> [0,0,1]
y_onehot = tf.keras.utils.to_categorical(y, num_classes=3)
print("Ejemplo de etiqueta one-hot:", y[0], "->", y_onehot[0])

X_train, X_test, y_train, y_test = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42, stratify=y)
print("Entrenamiento:", X_train.shape, "| Test:", X_test.shape)

## 2. Construir la red

Para clasificación **multiclase**: capa oculta ReLU y capa de salida con **3 neuronas softmax**
(probabilidad de cada especie). Pérdida: **categorical crossentropy**.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

modelo = Sequential([
    Input(shape=(4,)),
    Dense(8, activation="relu"),
    Dense(3, activation="softmax"),
])
modelo.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
modelo.summary()

## 3. Entrenar

In [ ]:
historia = modelo.fit(X_train, y_train, epochs=150, batch_size=8,
                      validation_split=0.1, verbose=0)
acc = modelo.evaluate(X_test, y_test, verbose=0)[1]
print("Entrenamiento terminado.")
print("Accuracy en test: %.1f %%" % (acc * 100))

## 4. Evaluación

`predict` devuelve, para cada flor, la **probabilidad de cada clase** (gracias a softmax suman 1).
La predicción es la clase con mayor probabilidad (`argmax`).

In [ ]:
probs = modelo.predict(X_test, verbose=0)
y_pred = probs.argmax(axis=1)
y_true = y_test.argmax(axis=1)

# Ejemplo: una flor del test
i = 0
print("Probabilidades de la 1ª flor del test:")
for nombre, pr in zip(iris.target_names, probs[i]):
    print(f"  {nombre:12s}: {pr*100:5.1f} %")
print("Predicho:", iris.target_names[y_pred[i]], "| Real:", iris.target_names[y_true[i]])

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=iris.target_names).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Matriz de confusión (3 clases)"); plt.tight_layout(); plt.show()

print(classification_report(y_true, y_pred, target_names=iris.target_names))

## Experimenta tú

1. Cambia la activación de salida a `sigmoid` y la pérdida: ¿empeora? (softmax es lo adecuado para multiclase disjunta).
2. Aumenta las neuronas de la capa oculta o las épocas.
3. Prueba `sparse_categorical_crossentropy` usando `y` (enteros) en lugar del one-hot.

## Resumen

- Clasificación multiclase: salida **softmax** (n neuronas) + **categorical crossentropy** + etiquetas **one-hot**.
- `softmax` reparte la probabilidad entre las clases (suman 1); `argmax` da la clase final.
- La **matriz de confusión** y el `classification_report` evalúan el modelo por clase.

Has construido clasificadores binario y multiclase: el mismo esqueleto de Keras, cambiando la salida y la
pérdida según el problema. En la próxima clase pasaremos a la **regresión** (predecir valores continuos).